# Sentinel-1 steepness

The deforestation mask does not use the logistic steepness. `sd` and flattening come from the whole backscatter series. Steepness `a` only moves `min_sse` and `min_sse_t`.

The paper sets the logistic amplitude from the 95th and 5th percentiles, and uses a growth parameter in [-5, -2]. The study fixed `a = -2`. That dates the inflection, not the start of the drop. If the drop has fallen 10% of the way from the high level to the low level, the inflection is `log(1/0.9 - 1) / a` images later. Because `a` is negative, that delay is positive: 1.1 images at `a = -2`, and 0.44 images at `a = -5`. One relative orbit is 12 days, so that is about 13 days versus 5 days. The 10% to 90% width is twice the delay. At `a = -2` that is the paper's two-image interval.

[`udf/logistic_curve_sse.py`](../udf/logistic_curve_sse.py) is that model. `k = p95 - p05`, the low level is `p05`, the curve is centered on an 11-image window, and `steepness_parameter` defaults to -2. `p05` and `p95` are taken from the whole series. The saved `output-udp/openEO.tif` has no time series, so the real `min_sse_t` cannot be refit here. The series below are synthetic, and the fit is the same sum of squares as the UDF.

The last cell serves a local page. Open the printed URL. The notebook renderer does not paint the figure. The fitted window usually locks onto the middle of the drop, so the detected image often stays put while `a` changes. What does move is the onset implied by the paper's delay: detected image minus that delay. `min_sse` moves as well. A slow ramp of many images is still dated near its middle; a steeper `a` does not pull that date back to the start.


In [1]:
import math
from pathlib import Path

import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import numpy as np

REPEAT_DAYS = 12
WINDOW = 11
N_IMAGES = 80
ONSET = 40.0
HIGH_DB = -8.0
LOW_DB = -14.0
GRID_NOISE_DB = 0.5
SEED = 0

FITTED_STEEPNESS = (-1.0, -2.0, -3.0, -5.0, -8.0)
RAMP_WIDTHS = (1, 2, 6, 12)
FIGURE_DIR = Path("s1-steepness")


def analytical_delay_images(steepness: float) -> float:
    """Images from the 10% decay point to the inflection.

    The paper writes log(1/0.9 - 1) / a. With a negative, the result is positive:
    the inflection is this many images after the 10% point.
    """
    return math.log(1.0 / 0.9 - 1.0) / steepness


print(f"{'a':>6}  {'delay images':>12}  {'delay days':>10}  {'10-90% images':>14}")
for steepness in FITTED_STEEPNESS:
    delay = analytical_delay_images(steepness)
    print(
        f"{steepness:6.1f}  {delay:12.2f}  {delay * REPEAT_DAYS:10.1f}  {2 * delay:14.2f}"
    )


     a  delay images  delay days   10-90% images
  -1.0          2.20        26.4            4.39
  -2.0          1.10        13.2            2.20
  -3.0          0.73         8.8            1.46
  -5.0          0.44         5.3            0.88
  -8.0          0.27         3.3            0.55


In [2]:
def logistic_series(time: np.ndarray, inflection: float, steepness: float, high: float, low: float) -> np.ndarray:
    amplitude = high - low
    return low + amplitude / (1.0 + np.exp(-steepness * (time - inflection)))


def linear_series(time: np.ndarray, onset: float, width: float, high: float, low: float) -> np.ndarray:
    progress = np.clip((time - onset) / width, 0.0, 1.0)
    return high + (low - high) * progress


def sliding_sse(series: np.ndarray, steepness: float, window: int = WINDOW) -> dict:
    """Same score as udf/logistic_curve_sse.py.

    p05 and p95 are percentiles of the whole series. The logistic is centered on
    each full window of `window` images. Incomplete edge windows are left as NaN,
    which is what skipna=False does in the UDF.
    """
    series = np.asarray(series, dtype=float)
    low = float(np.quantile(series, 0.05))
    high = float(np.quantile(series, 0.95))
    amplitude = high - low
    half = window // 2
    offsets = np.arange(window, dtype=float)
    exponent = -1.0 * steepness * (offsets - half)
    predicted = low + amplitude / (1.0 + np.exp(exponent))
    score = np.full(series.size, np.nan)
    for center in range(half, series.size - half):
        segment = series[center - half : center + half + 1]
        score[center] = float(np.sum((segment - predicted) ** 2))
    detected = int(np.nanargmin(score))
    return {
        "detected": detected,
        "min_sse": float(score[detected]),
        "sse": score,
        "predicted": predicted,
        "low": low,
        "high": high,
        "half": half,
    }


In [3]:
def draw_series(ax, time, series, fit, onset, midpoint, title: str) -> None:
    half = fit["half"]
    detected = fit["detected"]
    ax.plot(time, series, color="#444444", lw=1)
    window_time = time[detected - half : detected + half + 1]
    ax.plot(window_time, fit["predicted"], color="#d62828", lw=2)
    ax.axvline(onset, color="#1b7a3a", ls="--", lw=1, label="onset")
    ax.axvline(midpoint, color="#4cc9f0", ls=":", lw=1, label="midpoint")
    ax.axvline(detected, color="#d62828", lw=1, label="detected")
    ax.set_title(title)
    ax.set_xlabel("image")
    ax.set_ylabel("dB")
    ax.legend(fontsize=7, loc="lower left")


rng = np.random.default_rng(SEED)
time = np.arange(N_IMAGES, dtype=float)
noise = rng.normal(size=N_IMAGES)

records = []


def evaluate(name: str, truth: np.ndarray, onset: float, midpoint: float, noisy: np.ndarray) -> None:
    for steepness in FITTED_STEEPNESS:
        fit = sliding_sse(noisy, steepness)
        records.append(
            {
                "name": name,
                "steepness": steepness,
                "detected": fit["detected"],
                "onset": onset,
                "midpoint": midpoint,
                "error_onset": fit["detected"] - onset,
                "error_midpoint": fit["detected"] - midpoint,
                "min_sse": fit["min_sse"],
            }
        )


for true_steepness in (-2.0, -5.0):
    inflection = ONSET + analytical_delay_images(true_steepness)
    truth = logistic_series(time, inflection, true_steepness, HIGH_DB, LOW_DB)
    evaluate(
        f"logistic a={true_steepness:.0f}",
        truth,
        onset=ONSET,
        midpoint=inflection,
        noisy=truth + GRID_NOISE_DB * noise,
    )

for width in RAMP_WIDTHS:
    truth = linear_series(time, ONSET, width, HIGH_DB, LOW_DB)
    evaluate(
        f"ramp {width} img",
        truth,
        onset=ONSET,
        midpoint=ONSET + width / 2.0,
        noisy=truth + GRID_NOISE_DB * noise,
    )

print(f"{'truth':<18} {'a':>5} {'detected':>9} {'err onset':>10} {'err mid':>8} {'min_sse':>8}")
for row in records:
    print(
        f"{row['name']:<18} {row['steepness']:5.1f} {row['detected']:9d} "
        f"{row['error_onset']:10.2f} {row['error_midpoint']:8.2f} {row['min_sse']:8.2f}"
    )

spike_truth = linear_series(time, ONSET, 2, HIGH_DB, LOW_DB)
spike = spike_truth.copy()
spike[12:14] += 4.0
clean_fit = sliding_sse(spike_truth, -2.0)
spike_fit = sliding_sse(spike, -2.0)
print(
    f"spike at images 12-13: detected {clean_fit['detected']} without, "
    f"{spike_fit['detected']} with, min_sse {clean_fit['min_sse']:.2f} -> {spike_fit['min_sse']:.2f}"
)


truth                  a  detected  err onset  err mid  min_sse
logistic a=-2       -1.0        41       1.00    -0.10     5.13
logistic a=-2       -2.0        41       1.00    -0.10     8.07
logistic a=-2       -3.0        41       1.00    -0.10    10.96
logistic a=-2       -5.0        41       1.00    -0.10    12.97
logistic a=-2       -8.0        41       1.00    -0.10    13.31
logistic a=-5       -1.0        41       1.00     0.56     7.76
logistic a=-5       -2.0        41       1.00     0.56     8.58
logistic a=-5       -3.0        40       0.00    -0.44     9.82
logistic a=-5       -5.0        40       0.00    -0.44    10.47
logistic a=-5       -8.0        40       0.00    -0.44    10.60
ramp 1 img          -1.0        41       1.00     0.50     9.75
ramp 1 img          -2.0        41       1.00     0.50     9.28
ramp 1 img          -3.0        41       1.00     0.50    10.65
ramp 1 img          -5.0        41       1.00     0.50    11.82
ramp 1 img          -8.0        41      

In [4]:
FIGURE_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(10, 6), dpi=100, layout="constrained")
examples = [
    ("logistic a=-2, fit -2", logistic_series(time, ONSET + analytical_delay_images(-2), -2, HIGH_DB, LOW_DB), -2.0, ONSET, ONSET + analytical_delay_images(-2)),
    ("logistic a=-2, fit -5", logistic_series(time, ONSET + analytical_delay_images(-2), -2, HIGH_DB, LOW_DB), -5.0, ONSET, ONSET + analytical_delay_images(-2)),
    ("ramp 6 img, fit -2", linear_series(time, ONSET, 6, HIGH_DB, LOW_DB), -2.0, ONSET, ONSET + 3),
    ("ramp 6 img, fit -5", linear_series(time, ONSET, 6, HIGH_DB, LOW_DB), -5.0, ONSET, ONSET + 3),
]
for ax, (title, truth, steepness, onset, midpoint) in zip(axes.ravel(), examples):
    fit = sliding_sse(truth, steepness)
    draw_series(ax, time, truth, fit, onset, midpoint, title)
fig.savefig(FIGURE_DIR / "examples.png")
plt.close(fig)

names = []
for row in records:
    if row["name"] not in names:
        names.append(row["name"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=100, layout="constrained")
for name in names:
    rows = [row for row in records if row["name"] == name]
    xs = [row["steepness"] for row in rows]
    axes[0].plot(xs, [row["error_onset"] for row in rows], marker="o", label=name)
    axes[1].plot(xs, [row["error_midpoint"] for row in rows], marker="o", label=name)
ref_a = np.array(FITTED_STEEPNESS)
axes[0].plot(ref_a, [analytical_delay_images(a) for a in ref_a], color="black", ls="--", label="paper delay")
for ax, ylabel in zip(axes, ("detected − onset (images)", "detected − midpoint (images)")):
    ax.axhline(0, color="#888888", lw=0.8)
    ax.set_xlabel("fitted steepness a")
    ax.set_ylabel(ylabel)
    ax.invert_xaxis()
axes[0].legend(fontsize=7, loc="center right")
fig.savefig(FIGURE_DIR / "date-error.png")
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4), dpi=100, layout="constrained")
for name in names:
    rows = [row for row in records if row["name"] == name]
    ax.plot([row["steepness"] for row in rows], [row["min_sse"] for row in rows], marker="o", label=name)
ax.set_xlabel("fitted steepness a")
ax.set_ylabel("min SSE (dB²)")
ax.invert_xaxis()
ax.legend(fontsize=7, loc="best")
fig.savefig(FIGURE_DIR / "min-sse.png")
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), dpi=100, layout="constrained", sharex=True)
draw_series(axes[0], time, spike_truth, clean_fit, ONSET, ONSET + 1, "ramp 2 img, no spike, fit a=-2")
draw_series(axes[1], time, spike, spike_fit, ONSET, ONSET + 1, "same series plus a +4 dB spike at images 12–13")
fig.savefig(FIGURE_DIR / "spike.png")
plt.close(fig)

for path in ("examples.png", "date-error.png", "min-sse.png", "spike.png"):
    print(FIGURE_DIR.resolve() / path)


/home/georg/code/LEON-P6/openeo/s1-processing/s1-steepness/examples.png
/home/georg/code/LEON-P6/openeo/s1-processing/s1-steepness/date-error.png
/home/georg/code/LEON-P6/openeo/s1-processing/s1-steepness/min-sse.png
/home/georg/code/LEON-P6/openeo/s1-processing/s1-steepness/spike.png


In [5]:
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

from IPython.display import HTML, display

_PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>logistic steepness</title>
<style>
body { font: 14px/1.4 sans-serif; margin: 12px 16px; color: #fff; background: #1e1e1e; }
label { display: grid; grid-template-columns: 160px 280px 72px; gap: 8px; align-items: center; margin: 6px 0; }
output { font-variant-numeric: tabular-nums; color: #fff; background: #111; padding: 1px 6px; border-radius: 3px; }
img { max-width: 100%; margin-top: 8px; background: #fff; }
#status { min-height: 1.2em; color: #ccc; }
</style>
</head>
<body>
<label>steepness a <input id="a" type="range" min="-8" max="-0.5" step="0.1" value="-2"><output id="a_v"></output></label>
<label>drop length img <input id="width" type="range" min="1" max="24" step="1" value="2"><output id="width_v"></output></label>
<label>noise dB <input id="noise" type="range" min="0" max="2" step="0.05" value="0.5"><output id="noise_v"></output></label>
<div id="status"></div>
<img id="series" alt="synthetic series">
<script>
const fields = [
  ["a", "a_v", "steepness"],
  ["width", "width_v", "drop_length"],
  ["noise", "noise_v", "noise_db"],
];
let seq = 0;
let timer = 0;
function refresh() {
  const my = ++seq;
  const params = new URLSearchParams();
  for (const [id, out, name] of fields) {
    const el = document.getElementById(id);
    document.getElementById(out).value = el.value;
    params.set(name, el.value);
  }
  document.getElementById("status").textContent = "drawing\u2026";
  const next = new Image();
  next.onload = () => {
    if (my !== seq) return;
    document.getElementById("series").src = next.src;
    document.getElementById("status").textContent = "";
  };
  next.onerror = () => {
    if (my === seq) document.getElementById("status").textContent = "draw failed";
  };
  next.src = "/series.png?" + params.toString() + "&n=" + my;
}
for (const [id] of fields) {
  document.getElementById(id).addEventListener("input", () => {
    clearTimeout(timer);
    timer = setTimeout(refresh, 200);
  });
}
refresh();
</script>
</body>
</html>
"""

_RENDER_LOCK = threading.Lock()
_PAGE_NOISE = np.random.default_rng(1).normal(size=N_IMAGES)


def render_series_png(steepness: float, drop_length: float, noise_db: float) -> bytes:
    truth = linear_series(time, ONSET, drop_length, HIGH_DB, LOW_DB)
    series = truth + noise_db * _PAGE_NOISE
    fit = sliding_sse(series, steepness)
    detected = fit["detected"]
    bias_images = detected - ONSET
    paper = analytical_delay_images(steepness)
    implied_onset = detected - paper
    fig, ax = plt.subplots(figsize=(9, 4.2), dpi=100, layout="constrained")
    ax.plot(time, truth, color="#bbbbbb", lw=1, label="truth")
    ax.plot(time, series, color="#444444", lw=1, label="with noise")
    half = fit["half"]
    window_time = time[detected - half : detected + half + 1]
    ax.plot(window_time, fit["predicted"], color="#d62828", lw=2, label="fitted window")
    ax.axvline(ONSET, color="#1b7a3a", ls="--", lw=1, label="true onset")
    ax.axvline(implied_onset, color="#f4a261", ls="--", lw=1, label="implied onset")
    ax.axvline(detected, color="#d62828", lw=1, label="detected")
    ax.set_title(
        f"detected image {detected} ({detected * REPEAT_DAYS:.0f} days)    "
        f"bias vs onset {bias_images:+.1f} img ({bias_images * REPEAT_DAYS:+.0f} days)\n"
        f"implied onset {implied_onset:.1f} ({implied_onset * REPEAT_DAYS:.0f} days)    "
        f"min_sse {fit['min_sse']:.1f} dB²    "
        f"paper delay {paper:.2f} img"
    )
    ax.set_xlabel("image")
    ax.set_ylabel("dB")
    ax.legend(fontsize=8, loc="lower left")
    buf_fig = __import__("io").BytesIO()
    fig.savefig(buf_fig, format="png")
    plt.close(fig)
    return buf_fig.getvalue()


class _SteepnessHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt: str, *args) -> None:
        return

    def _send(self, code: int, body: bytes, content_type: str) -> None:
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self) -> None:
        parsed = urlparse(self.path)
        if parsed.path == "/":
            self._send(200, _PAGE.encode(), "text/html; charset=utf-8")
            return
        if parsed.path != "/series.png":
            self._send(404, b"not found", "text/plain; charset=utf-8")
            return
        query = parse_qs(parsed.query)
        try:
            steepness = float(query["steepness"][0])
            drop_length = float(query["drop_length"][0])
            noise_db = float(query["noise_db"][0])
        except (KeyError, ValueError, IndexError):
            self._send(400, b"bad parameters", "text/plain; charset=utf-8")
            return
        if drop_length < 1:
            self._send(400, b"drop length must be at least 1", "text/plain; charset=utf-8")
            return
        try:
            with _RENDER_LOCK:
                png = render_series_png(steepness, drop_length, noise_db)
        except Exception as exc:
            self._send(500, str(exc).encode(), "text/plain; charset=utf-8")
            return
        self._send(200, png, "image/png")


class _ReusableServer(ThreadingHTTPServer):
    allow_reuse_address = True


def serve_steepness() -> str:
    global _SERVER
    try:
        server = _SERVER
    except NameError:
        server = None
    if server is None:
        httpd = _ReusableServer(("127.0.0.1", 8766), _SteepnessHandler)
        threading.Thread(target=httpd.serve_forever, daemon=True).start()
        _SERVER = httpd
        server = httpd
    host, port = server.server_address
    return f"http://{host}:{port}/"


url = serve_steepness()
print(url)
display(HTML(
    f'<p><a href="{url}" target="_blank">{url}</a></p>'
    f'<iframe src="{url}" title="steepness" width="900" height="640" '
    f'style="border:1px solid #ccc;max-width:100%"></iframe>'
))


http://127.0.0.1:8766/
